In [29]:
import numpy as np
import nestpy as nestpy
from matplotlib import pyplot as plt

import matplotlib as mpl
mpl.rcParams['lines.linewidth'] = 3
mpl.rcParams['font.size'] = 16

SHOWPLOTS=False
SAVEFIGURES=False



# Energy resolution modeling

A simple energy resolution model can be built by propagating errors through the energy reconstruction equation, assuming the various sources of noise are uncorrelated.

\begin{equation}
E = W \, N_q = W(N_e + N_{ph})
\end{equation}
Where $W = 13.7$~eV, and $N_e$, $N_{ph}$, and $N_q$ are the numbers of ionized electrons, scintillation photons, and total quanta (photons + electrons), respectively.

Assuming the uncertainties/errors on $N_e$ and $N_{ph}$ are independent, we can write:
\begin{equation}
\frac{\sigma_E}{E} = \frac{\sqrt{\sigma_{N_e}^2 + \sigma_{N_{ph}}^2 + F \cdot N_q} } {N_q}
\end{equation}

The $F \cdot N_q$ term is the Fano fluctuations in the total number of quanta produced; the predicted value of F in liquid xenon is 0.06. The other two terms are the noise stemming from the measurement of the number of electrons and the number of photons. Let's look at each of these separately.

### Noise sources in $N_e$

Sources of noise in the measurement of the number of electrons include:
- Finite electron lifetime $\tau$ (introduces both statistical fluctuations and a calibration error)
  - Efficiency expressed as an exponential depending on drift time: $\epsilon_{el} = e^{-t/\tau}$ 
- Finite electron extraction efficiency (introduces statistical fluctuations)
  - Efficiency expressed as a constant extraction probability $\epsilon_{eee}$
- Nonuniform single-electron gain response $G_{SE}$ (introduces a calibration error, maybe folded into the eee above?)
  - Simplest model is to add in quadrature a gaussian smearing term $\sigma_{G_{SE}}$
- S2 response noise from statistical fluctuations in the number of photons generated per electron
  - Can model as a gaussian smearing with width $\sigma_{pe/e}$. The overall noise added to a signal of size $N_e$ is $\sqrt{N_e} \sigma_{pe/e}$
- Electronics noise in PMT waveforms (statisical fluctuations, I will assme this is folded into the above)
- Saturation correction noise?

We can express $N_e$ in terms of various measured quantities and efficiencies: 

\begin{equation}
N_e = \frac{N_{e,meas}}{\epsilon_{el} \, \epsilon_{eee}} = \left(\frac{S2}{g_2}\right)\frac{1}{ \epsilon_{el} } =  \frac{S2}{G_{SE}} \times \frac{1}{\epsilon_{eee} \, \epsilon_{el}}
\end{equation}
where $S2$ is the measured light in units of photoelectrons and $g_2$ is defined as the product of the extraction efficiency and the single electron gain response: $g_2 = \epsilon_{eee} \times G_{SE}$.

The complicated term here is $\epsilon_{el}$, beacuse we typically think of it in terms of the electron lifetime, which will have asymmetric uncertainties because it lives in the denominator of an exponent. To make it easier, we can quantify the uncertainty in the inverse of this quantity, i.e. $\lambda = 1/\tau$. The variance term is then:
\begin{equation}
\left(\frac{\partial N_e}{\partial \epsilon_{el}}\right)^2 \left(\frac{\partial \epsilon_{el}}{\partial \lambda}\right)^2 \sigma_\lambda^2 = 
\left(\frac{N_{e,meas}}{\epsilon_{el}^2 \, \epsilon_{eee}}\right)^2 \left(-t e^{-t/\tau}\right)^2  \sigma_\lambda^2 = 
\left(\frac{N_e}{\epsilon_{el}}\right)^2 t^2 \epsilon_{el}^2 \sigma_\lambda^2 = 
N_e^2 t^2 \sigma_\lambda^2
\end{equation}


The errors/noise for $N_{e,meas}$ can be written out explicitly, where we account for the fact that two successive independent binomial processes can be combined into one:
\begin{equation}
\sigma^2_{N_{e,meas}} = N_e (\epsilon_{el} \, \epsilon_{eee}) (1 - \epsilon_{el} \, \epsilon_{eee}) + N_{e,meas}^2 \sigma^2_{G_{SE}} + N_{e,meas} \sigma_{pe/e}^2
\end{equation}

The first term models binomial fluctuations in the number of electrons that are either captured by impurities during drift or left behind at the liquid surface during extraction. The second term models mis-calibration of the single-electron gain as a gaussian random error added to the measurement (the variance should be proportional to the square of number of measured electrons, $N_{e,meas}$). The third term models the fluctuations in the number of photoelectrons per electron. The std. dev. should scale like the square root of $N_{e,meas}$, so that the relative error is $\sigma/\sqrt{N}$. 




### Noise sources in the light channel

The light measurement can be written out similarly to the charge:
\begin{equation}
N_{ph} = \frac{N_{ph,meas}}{\epsilon_{ph}} = \frac{S1}{g_1}
\end{equation}

Here the gain factor $g_1$ is the average light collection efficiency.

Sources of noise in the measurement of the number of $S1$ photons are:
- Binomial fluctuations in the photon collection efficiency, $\epsilon_{ph}$
- Imperfect calibration of the light collection efficiency map $\sigma_{\epsilon_{ph}}$
- Electronics noise in the photosensors(?)
- Fluctuations in how many photoelectrons per electron (at 175nm, get 2pe/ph ~20% of the time)

The third bullet and fourth bullets are likely to be negligible. If we omit them, we get:

\begin{equation}
\sigma_{N_{ph}}^2 = \left(\frac{1}{\epsilon_{ph}}\right)^2 N_{ph} \epsilon_{ph} (1 - \epsilon_{ph}) + N_{ph}^2 \sigma_{\epsilon_{ph}}^2
\end{equation}
where the first term models binomial fluctuations in the light collection and the second term models the error in the calibration being propagated into the measurement of energy across the entire detector


### Complete resolution model
Putting all of this together, and writing it out explicitly in terms of the various parameters, gives:
\begin{equation}
\begin{aligned}
\frac{\sigma_E}{E}  = \frac{\sqrt{\textcolor{green}{\sigma_{N_e}^2} + \textcolor{blue}{\sigma_{N_{ph}}^2} + F \cdot N_q} } {N_q}
\end{aligned}
\end{equation}

\begin{equation}
\begin{aligned}
\frac{\sigma_E}{E}  = \frac{ \sqrt{ 
                                \textcolor{green}{
                                \left( \frac{1}{\epsilon_{el} \, \epsilon_{eee}} \right)^2
                                \left(N_e (\epsilon_{el} \, \epsilon_{eee}) (1 - \epsilon_{el} \, \epsilon_{eee}) + 
                                N_{e,meas}^2 \sigma^2_{G_{SE}} + 
                                N_{e,meas} \sigma_{pe/e}^2 \right) + 
                                N_e^2 t^2 \sigma_\lambda^2
                                }    
                                +
                                \textcolor{blue}{
                                \frac{1}{\epsilon_{ph}}N_{ph}(1 - \epsilon_{ph}) + 
                                N_{ph}^2 \sigma_{\epsilon_{ph}}^2
                                }
                                +
                                F \cdot N_q}   
                            }{N_q}
\end{aligned}
\end{equation}


My initial parameters are:

| Parameter | Meaning | Default value |
| --- | --- | --- |
| $E$ | Energy of the event at $Q_{\beta\beta}$ | $2.457$ MeV |
| $W$ | Work function / energy per quanta | $11.5$ eV |
| $N_q$ | Total number of quanta produced | $E/W$ |
| $N_e$ | Average number of ionized electrons produced in the liquid | $111000$ |
| $N_{ph}$ | Average number of scintillation photons produced in the liquid | $N_q - N_e$ |
| $t$ | Average drift time in the detector | $3000\,\mathrm{mm} / 1.7(\mathrm{mm}/\mu\mathrm{s})$ |
| $\tau$ | Electron lifetime; used to compute $\epsilon_{el} = e^{-t/\tau}$ | $10,000\,\mu\mathrm{s}$ |
| $\epsilon_{eee}$ | Extraction efficiency | $0.7$ |
| $\epsilon_{ph}$ | Average light collection efficiency | $0.1$ |
| $\sigma_\lambda$ | Fractional broadening due to calibration uncertainty/error in $1/\tau$ | $1\%$ |
| $\sigma_{G_{SE}}$ | Fractional broadening from miscalibration of the single-electron gain $G_{SE}$ | $0.5\%$ |
| $\sigma_{pe/e}$ | Fractional resolution for a single-electron signal, modeling statistical fluctuations in the SE gain | $20\%$ |
| $\sigma_{\epsilon_{ph}}$ | Fractional broadening from miscalibration of $\epsilon_{ph}$ | $0.5\%$ |
| $f$ | Fano factor | $0.06$ |

With these parameters, the predicted resolution is:
\begin{equation}
\frac{\sigma_E}{E} = 0.73\%
\end{equation}

This will change as we refine our choices for each of the above parameters.

In [30]:
def energy_resolution_term_by_term(
    n_e,
    n_q,
    epsilon_eee,
    sigma_lambda,
    drift_time,
    e_lifetime,
    sigma_gse,
    sigma_pe_per_e,
    epsilon_ph,
    sigma_epsilon_ph,
    fano_factor=0.06,
    verbose=False,
):
    """Compute the energy resolution term by term and optionally print a breakdown."""

    epsilon_el = np.exp(-drift_time / e_lifetime)
    n_e_meas = n_e * (epsilon_el * epsilon_eee)
    n_ph = n_q - n_e

    sigma2_binomial = n_e * (epsilon_el * epsilon_eee) * (1 - epsilon_el * epsilon_eee) / (epsilon_el * epsilon_eee) ** 2
    sigma2_gain = n_e_meas**2 * sigma_gse**2 / (epsilon_el * epsilon_eee) ** 2
    sigma2_pe = n_e_meas * sigma_pe_per_e**2 / (epsilon_el * epsilon_eee) ** 2
    sigma2_charge_meas = sigma2_binomial + sigma2_gain + sigma2_pe

    sigma2_charge_lifetime = n_e**2 * drift_time**2 * (sigma_lambda * 1/e_lifetime)**2
    sigma2_charge = (
          sigma2_charge_meas
        + sigma2_charge_lifetime
    )

    sigma2_light_binomial = (1/epsilon_ph)**2 * n_ph * epsilon_ph * (1 - epsilon_ph)
    sigma2_light_calibration = n_ph**2 * sigma_epsilon_ph**2
    sigma2_light = sigma2_light_binomial + sigma2_light_calibration

    sigma2_fano = fano_factor * n_q
    sigma2_total = sigma2_charge + sigma2_light + sigma2_fano
    resolution = np.sqrt(sigma2_total) / n_q

    error_terms = {
        "binomial electron error": np.sqrt(sigma2_binomial),
        "gain calibration error": np.sqrt(sigma2_gain),
        "pe/e error": np.sqrt(sigma2_pe),
        "charge error from the above in quadrature": np.sqrt(sigma2_charge_meas),
        "charge error from lifetime calibration": np.sqrt(sigma2_charge_lifetime),
        "total charge error": np.sqrt(sigma2_charge),
        "light binomial error": np.sqrt(sigma2_light_binomial),
        "light calibration error": np.sqrt(sigma2_light_calibration),
        "total light error": np.sqrt(sigma2_light),
        "fano error": np.sqrt(sigma2_fano),
        "total error": np.sqrt(sigma2_total),
    }

    fractional_error_terms = {
        name: 100 * err / n_q for name, err in error_terms.items()
    }

    if verbose:
        def _fmt(value):
            if isinstance(value, str):
                return value
            return f"{value:0.6g}"

        def _print_table(title, rows):
            headers = ["Quantity", "Value"]
            data = [[name, _fmt(value)] for name, value in rows]
            widths = [
                max(len(headers[i]), max(len(row[i]) for row in data))
                for i in range(2)
            ]
            header_line = " | ".join(headers[i].ljust(widths[i]) for i in range(2))
            divider = "-+-".join("-" * width for width in widths)
            print(title)
            print(header_line)
            print(divider)
            for row in data:
                print(" | ".join(row[i].ljust(widths[i]) for i in range(2)))
            print()

        param_rows = [
            ("n_e", n_e),
            ("n_e_meas", n_e_meas),
            ("n_ph", n_ph),
            ("n_q", n_q),
            ("epsilon_el", epsilon_el),
            ("epsilon_eee", epsilon_eee),
            ("sigma_lambda", sigma_lambda),
            ("drift_time", drift_time),
            ("e_lifetime", e_lifetime),
            ("sigma_gse", sigma_gse),
            ("sigma_pe_per_e", sigma_pe_per_e),
            ("epsilon_ph", epsilon_ph),
            ("sigma_epsilon_ph", sigma_epsilon_ph),
            ("fano_factor", fano_factor),
        ]
        error_rows = list(error_terms.items())
        frac_rows = [(name, f"{value:.6g}%") for name, value in fractional_error_terms.items()]

        _print_table("Parameters", param_rows)
        _print_table("Error terms", error_rows)
        _print_table("Fractional error terms", frac_rows)
        print(f"Energy resolution = {resolution:.6f}")

    return {
        "resolution": resolution,
        "error_terms": error_terms,
        "fractional_error_terms": fractional_error_terms,
        "variance_terms": {
            "charge": sigma2_charge,
            "light": sigma2_light,
            "fano": sigma2_fano,
            "total": sigma2_total,
        },
    }


In [31]:
E = 2.457
W = 11.5 # eV
n_q = E/W * 1e6
n_e = 111000
n_ph = n_q - n_e
e_lifetime = 10000 # us
drift_time = 3000. / 1.7 # us
epsilon_eee = 0.7
epsilon_ph = 0.1
fano_factor = 0.06
sigma_pe_per_e = 0.2

# Relative error terms in calibrated quantities:
sigma_lambda = 0.01 
sigma_gse = 0.005
sigma_epsilon_ph = 0.01

epsilon_el = np.exp(-drift_time / e_lifetime)
print(np.exp(-drift_time / e_lifetime) * epsilon_eee)
print(np.sqrt(n_e * (epsilon_el * epsilon_eee) * (1 - epsilon_el * epsilon_eee) / (epsilon_el * epsilon_eee) ** 2)/1e5)
print('Drift time: {:4.4} us'.format(drift_time))
print('Sigma lambda: {:4.4} us^-1'.format(sigma_lambda))
print('Relative error in lambda: {:4.4} %'.format(sigma_lambda * e_lifetime * 100))
print('Relative error due to lambda error: {:4.4} %'.format(np.sqrt(n_e**2 * (drift_time)**2 * sigma_lambda**2)/n_q * 100))

res_values = energy_resolution_term_by_term(
    n_e=n_e,
    n_q=n_q,
    epsilon_eee=epsilon_eee,
    sigma_lambda=sigma_lambda,
    drift_time=drift_time,
    e_lifetime=e_lifetime,
    sigma_gse=sigma_gse,
    sigma_pe_per_e=sigma_pe_per_e,
    epsilon_ph=epsilon_ph,
    sigma_epsilon_ph=sigma_epsilon_ph,
    fano_factor=fano_factor,
    verbose=True,
)

0.5867564026960999
0.002795990152268626
Drift time: 1.765e+03 us
Sigma lambda: 0.01 us^-1
Relative error in lambda: 1e+04 %
Relative error due to lambda error: 916.8 %
Parameters
Quantity         | Value   
-----------------+---------
n_e              | 111000  
n_e_meas         | 65130   
n_ph             | 102652  
n_q              | 213652  
epsilon_el       | 0.838223
epsilon_eee      | 0.7     
sigma_lambda     | 0.01    
drift_time       | 1764.71 
e_lifetime       | 10000   
sigma_gse        | 0.005   
sigma_pe_per_e   | 0.2     
epsilon_ph       | 0.1     
sigma_epsilon_ph | 0.01    
fano_factor      | 0.06    

Error terms
Quantity                                  | Value  
------------------------------------------+--------
binomial electron error                   | 279.599
gain calibration error                    | 555    
pe/e error                                | 86.9886
charge error from the above in quadrature | 627.509
charge error from lifetime calibration    | 195.

# Resolution vs. light collection efficiency $\epsilon_{ph}$

In [32]:
E = 2.457
W = 11.5 # eV
n_q = E/W * 1e6
n_e = 111000
n_ph = n_q - n_e
e_lifetime = 10000 # us
drift_time = 3000. / 1.7 # us
epsilon_eee = 0.7
epsilon_ph = 0.1
fano_factor = 0.06
sigma_pe_per_e = 0.2

LCE_scan = np.linspace(0.01, 0.20, 100)

results_vs_LCE = energy_resolution_term_by_term(
    n_e=n_e,
    n_q=n_q,
    epsilon_eee=epsilon_eee,
    sigma_lambda=sigma_lambda,
    drift_time=drift_time,
    e_lifetime=e_lifetime,
    sigma_gse=sigma_gse,
    sigma_pe_per_e=sigma_pe_per_e,
    epsilon_ph=LCE_scan,
    sigma_epsilon_ph=sigma_epsilon_ph,
    fano_factor=fano_factor,
    verbose=False,
)

if SHOWPLOTS:
    plt.plot(LCE_scan*100, results_vs_LCE["resolution"]*100, label=r"")
    plt.xlabel("Light collection effciency $\epsilon_{ph}$ [%]")
    plt.ylabel(r"Overall energy resolution at $Q_{\beta\beta}$ [%]")
    # plt.title(r"Energy resolution vs error term value")
    # plt.legend(framealpha=1.)
    plt.grid(which='both', alpha=0.5)
    plt.xlim(0.,20.)
    plt.ylim(0.3,1.2)
    if SAVEFIGURES:
        plt.savefig('Eres vs LCE.png',dpi=200,bbox_inches='tight')



# Resolution vs. calibration precision

There are three parameters in our model that correspond to the fractional precision with which we can calibrate different aspects of the experiment:
 - $\sigma_{\epsilon_{ph}}$, the relative precision of the light map calibration, which varies as a function of $(x,y,z)$
 - $\sigma_{G_{SE}}$, the relative precision of the single electron gain, which varies as as a function of $(x,y)$
 - $\sigma_{\lambda}$, the relative precision of our measurement of the inverse electron lifetime ($\lambda = 1/\tau_e$)

 We can explicitly study the impact of these three parameters on the energy resolution:

 

In [33]:
E = 2.457
W = 11.5 # eV
n_q = E/W * 1e6
n_e = 111000
n_ph = n_q - n_e
e_lifetime = 10000 # us
drift_time = 3000. / 1.7 # us
epsilon_eee = 0.7
epsilon_ph = 0.1
fano_factor = 0.06
sigma_pe_per_e = 0.2

# Relative error terms in calibrated quantities (default values)
sigma_lambda = 0.01 
sigma_gse = 0.005
sigma_epsilon_ph = 0.01

# Values to scan across
sigma_ph_scan = np.linspace(0., 1.5/100, 100)
sigma_GSE_scan = np.linspace(0., 1.5/100, 100)
sigma_lambda_scan = np.linspace(0.,1.5/100,100)


results_by_sigma_ph_scan = energy_resolution_term_by_term(
    n_e=n_e,
    n_q=n_q,
    epsilon_eee=epsilon_eee,
    sigma_lambda=sigma_lambda,
    drift_time=drift_time,
    e_lifetime=e_lifetime,
    sigma_gse=sigma_gse,
    sigma_pe_per_e=sigma_pe_per_e,
    epsilon_ph=epsilon_ph,
    sigma_epsilon_ph=sigma_ph_scan,
    fano_factor=fano_factor,
    verbose=False,
)

results_by_sigma_GSE_scan = energy_resolution_term_by_term(
    n_e=n_e,
    n_q=n_q,
    epsilon_eee=epsilon_eee,
    sigma_lambda=sigma_lambda,
    drift_time=drift_time,
    e_lifetime=e_lifetime,
    sigma_gse=sigma_GSE_scan,
    sigma_pe_per_e=sigma_pe_per_e,
    epsilon_ph=epsilon_ph,
    sigma_epsilon_ph=sigma_epsilon_ph,
    fano_factor=fano_factor,
    verbose=False,
)

results_by_sigma_lambda_scan = energy_resolution_term_by_term(
    n_e=n_e,
    n_q=n_q,
    epsilon_eee=epsilon_eee,
    sigma_lambda=sigma_lambda_scan,
    drift_time=drift_time,
    e_lifetime=e_lifetime,
    sigma_gse=sigma_gse,
    sigma_pe_per_e=sigma_pe_per_e,
    epsilon_ph=epsilon_ph,
    sigma_epsilon_ph=sigma_epsilon_ph,
    fano_factor=fano_factor,
    verbose=False,
)


import matplotlib as mpl
mpl.rcParams['lines.linewidth'] = 3

if SHOWPLOTS:
    plt.plot(sigma_ph_scan*100, results_by_sigma_ph_scan["resolution"]*100, label=r"Varying $\sigma_{\epsilon_{ph}}$ ($\sigma_{GSE}$ = 0.5%, $\sigma_{\lambda}$ = 1%)")
    plt.plot(sigma_GSE_scan*100, results_by_sigma_GSE_scan["resolution"]*100, label=r"Varying $\sigma_{GSE}$ ($\sigma_{\epsilon_{ph}}$ = 1%, $\sigma_{\lambda}$ = 1%)")
    plt.plot(sigma_lambda_scan*100, results_by_sigma_lambda_scan["resolution"]*100, label=r"Varying $\sigma_{\lambda}$ ($\sigma_{\epsilon_{ph}}$ = 1%, $\sigma_{GSE}$ = 0.5%)")
    plt.xlabel("Error term value [%]")
    plt.ylabel(r"Overall energy resolution at $Q_{\beta\beta}$ [%]")
    # plt.title(r"Energy resolution vs error term value")
    plt.legend(framealpha=1.,fontsize=12)
    plt.grid(which='both', alpha=0.5)
    plt.xlim(0.,1.5)
    plt.ylim(0.3,1.2)
    if SAVEFIGURES:
        plt.savefig('Eres vs error terms.png',dpi=200,bbox_inches='tight')



In [34]:
nc = nestpy.NESTcalc(nestpy.detectors.VDetector())

# Function gets ER light and charge yields for a given energy and optional er_parameters list.
def standardQyLy(fields, m=None):
    if m is None:
        m = [-1 for _ in range(10)]

    fields = np.asarray(fields, dtype=float)
    qy_out = []
    ly_out = []
    energy = 2457.0  # keV

    for field in fields:
        er_yields = nc.GetYields(
            nestpy.interactions.gammaRay,
            energy=energy,
            drift_field=float(field),
            er_parameters=m,
        )
        qy_out.append(er_yields.ElectronYield / energy)
        ly_out.append(er_yields.PhotonYield / energy)

    return np.asarray(qy_out), np.asarray(ly_out)

In [35]:
def importEXO200Data():
    # EXO-200 data from https://arxiv.org/pdf/1908.04128
    # Table 3, page 13, yields measured from 208Tl line
    file = './exo200 yield measurements.csv'
    energy = 2615 # keV
    data = np.genfromtxt(file, delimiter=',', skip_header=1)
    fields = data[:, 0]
    n_e = 1000*data[:, 1]
    n_e_err = 1000*np.sqrt(data[:, 2]**2 + data[:, 3]**2) # combine stat and syst errors in quadrature
    n_ph = 1000*data[:, 4]
    n_ph_err = 1000*np.sqrt(data[:, 5]**2 + data[:, 6]**2) # combine stat and syst errors in quadrature
    
    qy = n_e / energy
    ly = n_ph / energy
    qy_err = n_e_err / energy
    ly_err = n_ph_err / energy
        
    return {'field': fields, 'qy': qy, 'ly': ly, 'qy_err': qy_err, 'ly_err': ly_err}




exo200_data = importEXO200Data()

field = np.linspace(10.,1000.,100)
qy, ly = standardQyLy(field)

if SHOWPLOTS:

    plt.plot(field, qy, color=(0.2,0.2,0.8), label="Qy (e/keV) from NEST")
    plt.errorbar(exo200_data['field'], exo200_data['qy'], yerr=exo200_data['qy_err'], fmt='o', color=(0.2,0.2,0.8), label="EXO-200 data", linewidth=1.5)
    plt.plot(field, ly, color=(0.8,0.2,0.2), label="Ly (ph/keV) from NEST")
    plt.errorbar(exo200_data['field'], exo200_data['ly'], yerr=exo200_data['ly_err'], fmt='o', color=(0.8,0.2,0.2), label="EXO-200 data", linewidth=1.5)
    # plt.plot(field, qy+ly, color=(0.2,0.8,0.2), label="Qy + Ly")

    plt.xlabel("Drift Field (V/cm)")
    plt.ylabel("Yield [quanta per keV]")
    plt.legend(framealpha=1,fontsize=12)
    plt.xlim(10.,1000.)
    plt.xscale('log')
    plt.ylim(0.,70.)
    plt.title('NEST gammaRay yields model at 2.457 MeV')
    plt.grid()
    if SAVEFIGURES:
        plt.savefig('Yields vs field w EXO200 data.png',dpi=200,bbox_inches='tight')

In [36]:
n_e = qy * 2457.
n_q = (n_e + ly * 2457.)[-1]
n_ph = ly * 2457.

print('Field = {:6.6}'.format(field[9]))
print("NEST-predicted N_e = {:6.6}".format(n_e[9]))
print("NEST-predicted N_ph = {:6.6}".format(n_ph[9]))
print("NEST-predicted N_q = {:6.6}".format(n_q))
print("NEST-predicted W = {:6.6} eV".format(2457./n_q*1000))


results_by_field_scan = energy_resolution_term_by_term(
    n_e=n_e,
    n_q=n_q,
    epsilon_eee=epsilon_eee,
    sigma_lambda=sigma_lambda,
    drift_time=drift_time,
    e_lifetime=e_lifetime,
    sigma_gse=sigma_gse,
    sigma_pe_per_e=sigma_pe_per_e,
    epsilon_ph=epsilon_ph,
    sigma_epsilon_ph=sigma_epsilon_ph,
    fano_factor=fano_factor,
    verbose=False,
)

if SHOWPLOTS:
    plt.plot(field, results_by_field_scan['resolution']*100)
    plt.ylabel(r'Energy resolution at $Q_{\beta\beta}$ [%]')
    plt.xlabel('Drift field [V/cm]')
    plt.xscale('log')
    plt.xlim(10.,1000.)
    plt.ylim(0.3,1.2)
    plt.grid(which='both', alpha=0.5)
    if SAVEFIGURES:
        plt.savefig('Eres vs drift field.png',dpi=200,bbox_inches='tight')


Field =  100.0
NEST-predicted N_e = 86584.9
NEST-predicted N_ph = 96222.6
NEST-predicted N_q = 1.82807e+05
NEST-predicted W = 13.4404 eV
